In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
dataset = df.sample(n=200, random_state=42).copy()

In [4]:
# Lowercasing : 

dataset['review'] = dataset['review'].str.lower()
dataset.head()

,review,sentiment
33553,i really liked this summerslam due to the look...,positive
9427,not many television shows appeal to quite as m...,positive
199,the film quickly gets to a major chase scene w...,negative
12447,jane austen would definitely approve of this o...,positive
39489,expectations were somewhat high for me when i ...,negative


When regx is used and when BeautifulScoup ?

“I use regex when the HTML is simple, predictable, and performance is critical—like removing basic tags in large datasets using vectorized Pandas operations. However, for complex or nested HTML structures, BeautifulSoup is more reliable because regex cannot correctly parse hierarchical data.”

In [5]:
# Remove 'html' tags : using BeautifulScoup 

from  bs4 import BeautifulSoup

dataset['review'] = dataset['review'].apply(lambda x : BeautifulSoup(x , 'html.parser').get_text())

dataset.head()


,review,sentiment
33553,i really liked this summerslam due to the look...,positive
9427,not many television shows appeal to quite as m...,positive
199,the film quickly gets to a major chase scene w...,negative
12447,jane austen would definitely approve of this o...,positive
39489,expectations were somewhat high for me when i ...,negative


In [6]:
# Remove 'html' tags : using Regx(regular expression)

import re

def remove_html(text):
    if not isinstance(text, str):
        return text
    pattern = re.compile(r'<[^>]+>')
    return pattern.sub('', text)

dataset['review'] = dataset['review'].apply(remove_html)

In [7]:
# Remove 'URLs' : “I use the pattern https?://\S+|www\.\S+ to capture both HTTP/HTTPS and www-based URLs, removing everything until the next whitespace.”

def remove_url(text):
    if not isinstance(text, str):
        return text
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub('', text)

dataset['review'] = dataset['review'].apply(remove_url)

In [8]:
# Remove 'Punctuation' :

import string

def remove_punc(text):
    if not isinstance(text , str):
        return text
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

dataset['review'] = dataset['review'].apply(remove_punc)

In [9]:
dataset.head()

,review,sentiment
33553,i really liked this summerslam due to the look...,positive
9427,not many television shows appeal to quite as m...,positive
199,the film quickly gets to a major chase scene w...,negative
12447,jane austen would definitely approve of this o...,positive
39489,expectations were somewhat high for me when i ...,negative


In [10]:
# Chat word treatment : when we have this type of data :
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}

In [11]:
# Chat word treatment : ex : replace Gn with Good night 

def chat_conversion(text) :
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

dataset['review'] = dataset['review'].apply(chat_conversion)

In [12]:
chat_conversion('IMHO is the best')

'In My Honest/Humble Opinion is the best'

# Text Representation/ Feature Extracting

# 1. Bag Of Words

In [13]:
print((dataset['review'].str.strip() == "").sum())

0


In [14]:
# Bag of Words :

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()


In [15]:
bow = cv.fit_transform(dataset['review']) # there are too many hyperparameters so read it form documentation

In [16]:
# print(cv.vocabulary_)

In [17]:
print(bow[0].toarray())

[[0 0 0 ... 0 0 0]]


In [18]:
# again convert to dataframe

import pandas as pd

bow_df = pd.DataFrame(
    bow.toarray(),
    columns=cv.get_feature_names_out(),
    index=dataset.index
)

final_dataset = pd.concat([dataset['sentiment'], bow_df], axis=1)

In [19]:
final_dataset.head()

,sentiment,04,0that,10,100,100000,1010,1010dialogues,1010direction,1010music,...,yv,zalinsky,zany,zero,zest,ziploc,zombie,zombies,zomedy,zone
33553,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9427,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
199,negative,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12447,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39489,negative,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 2. N Grams 

In [20]:
dataset2 = df.sample(n=200, random_state=42).copy()


In [21]:
dataset2['review'] = dataset['review'].apply(remove_html)

In [22]:
dataset2['review'] = dataset['review'].str.lower()

In [23]:
dataset['review'] = dataset['review'].apply(remove_url)

In [24]:
dataset['review'] = dataset['review'].apply(remove_punc)

In [25]:
dataset['review'] = dataset['review'].apply(chat_conversion)

In [26]:
# N grams :

from sklearn.feature_extraction.text import CountVectorizer

cv2 = CountVectorizer(ngram_range = (3,3) )

In [27]:
bow2 = cv2.fit_transform(dataset2['review'])

In [28]:
print(bow2[0].toarray())


[[0 0 0 ... 0 0 0]]


In [29]:
# print(cv2.vocabulary_)

In [30]:
import pandas as pd

bow_df2 = pd.DataFrame(
    bow2.toarray(),
    columns=cv2.get_feature_names_out(),
    index=dataset.index
)

final_dataset2 = pd.concat([dataset2['sentiment'], bow_df2], axis=1)

In [31]:
final_dataset2.head()

,sentiment,04 supporters are,0that said she,10 for several,10 minutes and,10 minutes of,10 will see,100 million bc,100 original this,100000 the evidence,...,zombie movies unless,zombie why not,zombies and prove,zombies and young,zombies ever as,zombies have been,zombies small children,zombies with bolt,zomedy seriously thats,zone they can
33553,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9427,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
199,negative,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12447,positive,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39489,negative,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 3. TF-IDF Method : 

In [32]:
dataset3 = df.sample(n=200, random_state=42).copy()

In [33]:
dataset2['review'] = dataset['review'].apply(remove_html)
dataset2['review'] = dataset['review'].str.lower()
dataset['review'] = dataset['review'].apply(remove_url)
dataset['review'] = dataset['review'].apply(remove_punc)
dataset['review'] = dataset['review'].apply(chat_conversion)

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [35]:
tfidf = TfidfVectorizer()

tfidf_dataset = tfidf.fit_transform(df['review'])

In [36]:
print(tfidf.idf_)
print(tfidf.get_feature_names_out())

[ 6.63801473  5.70390616 11.1266511  ... 11.1266511  11.1266511
 11.1266511 ]
['00' '000' '00000000000' ... 'żmijewski' 'יגאל' 'כרמון']
